Day 1, Topic 6: Lab Notebook – The Byte Inspector

## 📘 Topic 6: Lab Notebook — The Byte Inspector

### What This Lab Teaches
This lab synthesizes **Topics 1–3** into one hands-on function. You'll combine:
- `dtype` and memory knowledge (Topic 1)
- `.view()` to reinterpret bytes (Topic 1)
- Understanding of what `.nbytes` tells you (Topic 2)
- Raw byte inspection with `.tobytes()` (Topic 1)

### The Task: `inspect_bytes(arr)`
You need to write a function that takes **any NumPy array** and gives you a complete picture of its memory representation from multiple angles.

### Core Tools You'll Need

| Tool | What It Does |
|------|-------------|
| `arr.shape` | Dimensions of the array |
| `arr.dtype` | Data type (e.g., `int16`, `float32`) |
| `arr.nbytes` | Total bytes = `arr.size × arr.itemsize` |
| `arr.tobytes().hex()` | Raw memory as a hexadecimal string |
| `arr.view(np.int8)` | Reinterpret every byte individually as a signed int |
| `arr.view(np.float32)` | Reinterpret bytes as 32-bit floats (needs `nbytes % 4 == 0`) |
| `arr.view(np.uint16)` | Reinterpret bytes as 16-bit unsigned ints (needs `nbytes % 2 == 0`) |
| `.tobytes().decode('ascii')` | Try to read raw bytes as ASCII text (fun bonus!) |

### Why the Divisibility Check?
You can only reinterpret bytes if the **total byte count divides evenly** by the new dtype's size:
```python
# arr has 8 bytes total
arr.view(np.float32)  # each float32 = 4 bytes → 8/4 = 2 elements ✓
arr.view(np.float64)  # each float64 = 8 bytes → 8/8 = 1 element  ✓
arr.view(np.float64)  # 7 bytes total → 7/8 = not whole → ❌ ValueError
```

### The ASCII Trick — Why It's Fun
If your array contains values in the ASCII range (0–127), decoding as text reveals the character representation. For example:
```python
np.array([65, 66, 67, 68], dtype=np.int16)
# Hex bytes: 41 00 42 00 43 00 44 00
# ASCII:      A  \0  B  \0  C  \0  D  \0
```
The `\0` (null bytes) appear because `int16` stores each value in 2 bytes, but ASCII 65 only needs 1 byte — the second byte is always 0 in little-endian systems.

### Little-Endian vs Big-Endian (Bonus Knowledge)
Most modern CPUs (x86/x64) are **little-endian**: the **least significant byte** is stored first.
```
int16 value 65 = 0x0041
Stored in memory: 41 00  ← least significant byte first (little-endian)
                  00 41  ← most significant byte first (big-endian)
```
This is why `arr.tobytes().hex()` gives `4100` for `np.int16([65])` instead of `0041`.

Write a function inspect_bytes(arr) that:

Prints the array's shape, dtype, and total bytes.

Prints the raw bytes as a hexadecimal string.

Prints the array viewed as np.int8 (1‑byte signed integers).

Prints the array viewed as np.float32 (if size is compatible).

Prints the array viewed as np.uint16 (if size is compatible).

(Bonus) Prints the ASCII interpretation of the bytes (for fun).

### 💡 Implementation Notes

**Checking compatibility before `.view()`:**
```python
if arr.nbytes % 4 == 0:
    print(arr.view(np.float32))
else:
    print("Cannot view as float32 — bytes not divisible by 4")
```

**Handling decode errors gracefully:**
```python
try:
    ascii_str = arr.tobytes().decode('ascii')
    print(f"ASCII: '{ascii_str}'")
except UnicodeDecodeError:
    print("ASCII: (contains non-printable bytes)")
```

**What the output tells you:**
- `int8` view: shows you each **individual byte** as a number (-128 to 127)
- `float32` view: shows what those bytes would mean if they were floats (usually gibberish — that's okay!)
- `uint16` view: useful when your data was originally unsigned 16-bit (audio samples, image pixels, etc.)
- ASCII: reveals if your numeric data happens to spell something readable

In [1]:
import numpy as np

### ✅ Solution — `inspect_bytes(arr)`
The function below implements all required parts plus the ASCII bonus:

In [44]:
def inspect_bytes(arr):
    print("1. Array shape: ", arr.shape)
    print("2. Array dtype: ", arr.dtype)
    print("3. Array total bytes: ", arr.nbytes)
    print("Tobytes: ", arr.tobytes().hex())
    view_int8 = arr.view(np.int8)
    print("4. Viewed as int8: ", view_int8)
    if arr.nbytes % 4 == 0:
        view_float32 = arr.view(np.float32)
        print("4. Viewed as float32: ", view_float32)
    else:
        print("Cannot view as float32: total bytes not divisible by 4")
    if arr.nbytes % 2 == 0:
        view_uint16 = arr.view(np.uint16)
        print("4. Viewed as uint16: ", view_uint16)
    else:
        print("Cannot view as uint16: total bytes not divisible by 2")
    try:
        ascii_str = arr.tobytes().decode('ascii')
        print(f"ASCII interpretation: '{ascii_str}'")
    except UnicodeDecodeError:
        print("ASCII interpretation: (contains non-printable bytes)")
    print("=" * 50)

In [48]:
arr1 = np.array([65, 66, 67, 68], dtype=np.int16)
inspect_bytes(arr1)

1. Array shape:  (4,)
2. Array dtype:  int16
3. Array total bytes:  8
Tobytes:  4100420043004400
4. Viewed as int8:  [65  0 66  0 67  0 68  0]
4. Viewed as float32:  [6.061234e-39 6.244908e-39]
4. Viewed as uint16:  [65 66 67 68]
ASCII interpretation: 'A B C D '


### 🔍 Reading the Output for `arr1 = np.array([65, 66, 67, 68], dtype=np.int16)`

```
shape:  (4,)          → 4 elements
dtype:  int16         → 2 bytes each
nbytes: 8             → 4 × 2 = 8 bytes total
hex:    4100420043004400  → 8 bytes shown as 16 hex chars
                           41=65, 00=pad, 42=66, 00=pad ...
int8 view: [65, 0, 66, 0, 67, 0, 68, 0]  → each int16 becomes 2 int8s
           (little-endian: value byte first, then 0x00)
float32:   [6.06e-39, 6.24e-39]  → meaningless floats, just showing bytes
uint16:    [65, 66, 67, 68]  → same values! (int16 and uint16 same for 0–32767)
ASCII:     'A\0B\0C\0D\0'  → ASCII A=65, B=66, C=67, D=68, with null padding
```

> 💡 **Key takeaway:** The bytes never changed. What changed is only how we **interpret** them. This is the essence of Topic 1 (dtype) in action.